# Ponderada ROS

### Importando bibliotecas e Preparando a Imagem

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

caminho_imagem = 'dog.jpg'

imagem_bgr_original = cv2.imread(caminho_imagem)

fator_escala = 4
imagem_bgr = imagem_bgr_original[::fator_escala, ::fator_escala]

print(f"Resolução antiga: {imagem_bgr_original.shape}")
print(f"Nova resolução: {imagem_bgr.shape}")

### Conversão da Imagem de BGR para Cinza

In [ ]:
def converter_para_cinza(img_bgr):

    b = img_bgr[:, :, 0]
    g = img_bgr[:, :, 1]
    r = img_bgr[:, :, 2]

    img_cinza = 0.299 * r + 0.587 * g + 0.114 * b

    return img_cinza.astype(np.uint8)

imagem_cinza = converter_para_cinza(imagem_bgr)

plt.imshow(imagem_cinza, cmap='gray')
plt.title("Imagem convertida para Cinza")
plt.axis('off')
plt.show()

### Realiza Suavização da Imagem usando Convulação com Kernel Gaussiano

In [ ]:
def convolucao_2d(imagem, kernel):
    img_h, img_w = imagem.shape
    k_h, k_w = kernel.shape
    pad_h, pad_w = k_h // 2, k_w // 2

    img_padded = np.zeros((img_h + 2 * pad_h, img_w + 2 * pad_w))
    img_padded[pad_h:img_h + pad_h, pad_w:img_w + pad_w] = imagem

    imagem_suavizada = np.zeros_like(imagem, dtype=np.float32)

    for i in range(img_h):
        for j in range(img_w):
            regiao = img_padded[i:i + k_h, j:j + k_w]
            imagem_suavizada[i, j] = np.sum(regiao * kernel)

    return imagem_suavizada.astype(np.uint8)

kernel_gaussian_3x3 = np.array(
    [[1, 2, 1],
     [2, 4, 2],
     [1, 2, 1]
    ], dtype=np.float32) / 16.0

imagem_suavizada = convolucao_2d(imagem_cinza, kernel_gaussian_3x3)

plt.imshow(imagem_suavizada, cmap='gray')
plt.title("Imagem Suavizada")
plt.axis('off')
plt.show()

### Compara as Imagens processadas

In [ ]:
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.title("Original")
plt.imshow(cv2.cvtColor(imagem_bgr, cv2.COLOR_BGR2RGB))
plt.axis('off')

plt.subplot(1, 3, 2)
plt.title("Tons de Cinza")
plt.imshow(imagem_cinza, cmap='gray')
plt.axis('off')

plt.subplot(1, 3, 3)
plt.title("Suavizada")
plt.imshow(imagem_suavizada, cmap='gray')
plt.axis('off')

plt.tight_layout()
plt.show()

### Calcula o Gradiente, Encontra a Margem e Adiciona Borda ao Redor da Imagem


In [ ]:
def detectar_bordas_sobel(imagem):

    sobel_x = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=np.float32)
    sobel_y = np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=np.float32)

    img_h, img_w = imagem.shape

    img_padded = np.zeros((img_h + 2, img_w + 2))
    img_padded[1:-1, 1:-1] = imagem

    grad_x = np.zeros_like(imagem, dtype=np.float32)
    grad_y = np.zeros_like(imagem, dtype=np.float32)

    for i in range(img_h):
        for j in range(img_w):
            regiao = img_padded[i:i+3, j:j+3]
            grad_x[i, j] = np.sum(regiao * sobel_x)
            grad_y[i, j] = np.sum(regiao * sobel_y)

    magnitude = np.sqrt(grad_x**2 + grad_y**2)
    magnitude = (magnitude / np.max(magnitude)) * 255
    return magnitude.astype(np.uint8)

### Limiarização da Imagem

Tudo que for maior ou igual ao limiar vira branco, o que for menor vira preto

In [ ]:
def aplicar_limiar(imagem, limiar=50):

    imagem_binaria = np.where(imagem >= limiar, 255, 0)
    return imagem_binaria.astype(np.uint8)


imagem_bordas = detectar_bordas_sobel(imagem_suavizada)
contorno_final = aplicar_limiar(imagem_bordas, limiar=50)

plt.figure(figsize=(15, 5))
plt.subplot(1, 2, 1)
plt.title("Bordas")
plt.imshow(imagem_bordas, cmap='gray')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.title("Contorno")
plt.imshow(contorno_final, cmap='gray')
plt.axis('off')
plt.show()

### Localiza todos os pixels brancos da imageme e Mapeia as Coordenadas

Organiza os Pontos para o Turtlesin sempre ir para o ponto mais perto de onde ele está

In [ ]:
def planejar_caminho_turtlesim(img_contorno, limite_turtlesim=11.0, margem=1.0, limiar_salto=1.0):
    linhas, colunas = np.where(img_contorno == 255)
    if len(linhas) == 0: return []

    pontos_originais = list(zip(linhas, colunas))
    pontos_ordenados = []
    ponto_atual = pontos_originais.pop(0)
    pontos_ordenados.append(ponto_atual)

    img_h, img_w = img_contorno.shape
    escala_util = limite_turtlesim - (2 * margem)

    def mapear(p):
        x = margem + (p[1] / (img_w - 1)) * escala_util
        y = margem + (1.0 - (p[0] / (img_h - 1))) * escala_util
        return (float(x), float(y))

    caminho_com_saltos = [mapear(ponto_atual)]

    while pontos_originais:
        matriz_pontos = np.array(pontos_originais)
        distancias = (matriz_pontos[:, 0] - ponto_atual[0])**2 + (matriz_pontos[:, 1] - ponto_atual[1])**2
        idx_proximo = np.argmin(distancias)
        dist_min = np.sqrt(distancias[idx_proximo])

        ponto_proximo = pontos_originais.pop(idx_proximo)
        coord_proxima = mapear(ponto_proximo)

        if dist_min > 5:
            caminho_com_saltos.append(None)

        caminho_com_saltos.append(coord_proxima)
        ponto_atual = ponto_proximo

    return caminho_com_saltos

caminho_robo = planejar_caminho_turtlesim(contorno_final)
print(f"Caminho gerado. Total de pontos: {len(caminho_robo)}")

### Visualização do Mapeamento para o Turtlesin

In [ ]:
plt.figure(figsize=(7, 7))
plt.title("Trajetória")

segmento_x, segmento_y = [], []
for p in caminho_robo:
    if p is None:
        plt.plot(segmento_x, segmento_y, color='blue', linewidth=2)
        segmento_x, segmento_y = [], []
    else:
        segmento_x.append(p[0])
        segmento_y.append(p[1])
plt.plot(segmento_x, segmento_y, color='blue', linewidth=2, label='Caminho do Turtlesin')

plt.xlim(0, 11); plt.ylim(0, 11)
plt.gca().set_aspect('equal')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

### Gera Arquivo com os Pontos para o Turtlesin percorrer

Lógica de quando o próximo ponto está longe ele não desenha na tela

In [ ]:
with open("caminho_turtlesim.txt", "w") as f:
    contador = 0
    for p in caminho_robo:
        if p is None:
            f.write("None,None\n")
            contador = 0
        else:
            if contador % 3 == 0:
                f.write(f"{p[0]},{p[1]}\n")
            contador += 1

print("Arquivo 'caminho_turtlesim.txt' gerado")